In [8]:
import sys
sys.path.insert(0, r'c:\My_Programs\Job Application Agent')
sys.path.insert(0, r'c:\My_Programs\Job Application Agent\backend')

## Seed Test Candidate (reuse existing or create new)

In [9]:
from backend.db.session import SessionLocal
from backend.db.models import User, CandidateProfile

def seed_candidate(email: str, name: str):
    db = SessionLocal()
    user = db.query(User).filter_by(email=email).first()
    if not user:
        user = User(email=email, full_name=name)
        db.add(user)
        db.commit()
        db.refresh(user)
    profile = db.query(CandidateProfile).filter_by(user_id=user.id).first()
    if not profile:
        profile = CandidateProfile(user_id=user.id)
        db.add(profile)
        db.commit()
        db.refresh(profile)
    db.close()
    return profile.id

id_prajwal = seed_candidate('prajwal@test.com', 'Prajwal Gupta')
id_sriharsha = seed_candidate('sriharsha@test.com', 'Sriharsha Vadavalli')
print('Prajwal profile id   :', id_prajwal)
print('Sriharsha profile id :', id_sriharsha)

Prajwal profile id   : 0ca68a04-12b1-40f0-bc8d-5b2227f97a5b
Sriharsha profile id : ce154f0c-f7f2-4beb-9d01-9977622eba8a


## Process & Store Resumes

In [10]:
from backend.app.services.resume_service import handle_resume_upload

resumes = [
    (r'..\sample_test_data\PrajwalGupta_Resume.pdf', id_prajwal),
    (r'..\sample_test_data\sriharsha-vadavalli.pdf', id_sriharsha),
]

for path, profile_id in resumes:
    with open(path, 'rb') as f:
        result = handle_resume_upload(f, candidate_profile_id=profile_id)
    parsed = result['parsed_data']
    print('=== Resume:', path.split('\\')[-1], '===')
    print(f"  Name       : {parsed['name']}")
    print(f"  Email      : {parsed['email']}")
    print(f"  Phone      : {parsed['phone']}")
    print(f"  Skills     : {parsed['skills']}")
    print(f"  Experience : {parsed['experience']}")
    print(f"  Education  : {parsed['education']}")
    print(f"  Projects   : {parsed['projects']}")
    print()

1. saving file...
2. processing resume...
3. creating DB entry with parsed data...
4. storing embedding...
5. storing structured data...


IntegrityError: (psycopg.errors.UniqueViolation) duplicate key value violates unique constraint "uq_skill_resume"
DETAIL:  Key (resume_id, skill_name)=(971c4140-f2ba-4ddc-bd9d-220df1e9a7a5, python) already exists.
[SQL: INSERT INTO skills (id, resume_id, skill_name) VALUES (%(id__0)s::UUID, %(resume_id__0)s::UUID, %(skill_name__0)s::VARCHAR), (%(id__1)s::UUID, %(resume_id__1)s::UUID, %(skill_name__1)s::VARCHAR), (%(id__2)s::UUID, %(resume_id__2)s::UUID, %(skill_name ... 486 characters truncated ... (skill_name__9)s::VARCHAR), (%(id__10)s::UUID, %(resume_id__10)s::UUID, %(skill_name__10)s::VARCHAR)]
[parameters: {'skill_name__0': 'python', 'id__0': UUID('d9a403bd-1410-443b-b58e-774172aae112'), 'resume_id__0': UUID('971c4140-f2ba-4ddc-bd9d-220df1e9a7a5'), 'skill_name__1': 'pandas', 'id__1': UUID('002e7f90-f1b5-4e06-ac87-e28964181a2a'), 'resume_id__1': UUID('971c4140-f2ba-4ddc-bd9d-220df1e9a7a5'), 'skill_name__2': 'git', 'id__2': UUID('3cc9b9a1-f6ea-495b-874b-32b5478824e7'), 'resume_id__2

## Vector Search

In [7]:
import sys
sys.path.insert(0, r'c:\My_Programs\Job Application Agent')
sys.path.insert(0, r'c:\My_Programs\Job Application Agent\backend')

from backend.app.services.search_service import search_resumes

results = search_resumes('python backend developer', top_k=5)
print(results)

[{'resume_id': '56ca4ba1-aa7d-418a-828e-607431c05ad7', 'name': 'Sriharsha Vadavalli', 'email': 'sriharsha@test.com', 'score': 0.8650684654712677}, {'resume_id': '971c4140-f2ba-4ddc-bd9d-220df1e9a7a5', 'name': 'Prajwal Gupta', 'email': 'prajwal@test.com', 'score': 0.8712777495384216}, {'resume_id': '6aae553c-2cf6-47a2-aaa6-fad285585df2', 'name': 'Test User', 'email': 'test@test.com', 'score': 0.8998029083013535}]
